In [72]:
# %%
# Analyze optimization metrics
# Open this file in Jupyter / VS Code and run cell by cell.

# %%
from __future__ import annotations

import re
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# %%
def parse_run_timestamp(run_dir_name: str) -> datetime | None:
    match = re.match(r"^(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})", run_dir_name)
    if match is None:
        return None

    try:
        return datetime.strptime(match.group(1), "%Y-%m-%d_%H-%M-%S")
    except ValueError:
        return None


def find_latest_run_dir(optimization_output_root: Path, index: int = 0) -> Path:
    if not optimization_output_root.exists():
        raise FileNotFoundError(f"OptimizationOutput folder does not exist: {optimization_output_root}")

    if index < 0:
        raise ValueError(f"index must be non-negative, got: {index}")

    candidate_run_dirs: list[dict[str, Any]] = []

    for child in optimization_output_root.iterdir():
        if not child.is_dir():
            continue

        metrics_csv_path = child / "metrics.csv"
        if not metrics_csv_path.exists():
            continue

        candidate_run_dirs.append(
            {
                "run_dir": child,
                "parsed_timestamp": parse_run_timestamp(child.name),
                "modified_time": metrics_csv_path.stat().st_mtime,
            }
        )

    if not candidate_run_dirs:
        raise FileNotFoundError(f"No run folders with metrics.csv found under: {optimization_output_root}")

    candidate_run_dirs.sort(
        key=lambda item: (
            item["parsed_timestamp"] is not None,
            item["parsed_timestamp"] if item["parsed_timestamp"] is not None else datetime.min,
            item["modified_time"],
        ),
        reverse=True,
    )

    if index >= len(candidate_run_dirs):
        raise IndexError(
            f"Requested index {index}, but only {len(candidate_run_dirs)} run(s) were found."
        )

    return candidate_run_dirs[index]["run_dir"]


def numeric_columns(dataframe: pd.DataFrame) -> list[str]:
    return [
        column_name
        for column_name in dataframe.columns
        if pd.api.types.is_numeric_dtype(dataframe[column_name])
    ]


def columns_containing(dataframe: pd.DataFrame, text: str) -> list[str]:
    if text.strip() == "":
        return []
    return [
        column_name
        for column_name in dataframe.columns
        if text.lower() in column_name.lower()
    ]


In [3]:
# %%
# ============================================================
# Gradient analyzer helper functions
# ============================================================

from pathlib import Path
from typing import Optional, Sequence

import colorsys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_latest_run_dir(
        optimization_output_root: Path,
        index: int = 0,
) -> Path:
    optimization_output_root = Path(optimization_output_root)

    if not optimization_output_root.exists():
        raise FileNotFoundError(
            f"Optimization output root does not exist: {optimization_output_root}"
        )

    candidates = [
        p for p in optimization_output_root.iterdir()
        if p.is_dir()
    ]

    if not candidates:
        raise FileNotFoundError(
            f"No run directories found under: {optimization_output_root}"
        )

    candidates = sorted(
        candidates,
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    if index < 0 or index >= len(candidates):
        raise IndexError(
            f"run_index={index} is out of range. Found {len(candidates)} runs."
        )

    return candidates[index]


def load_per_point_metrics(
        optimization_output_root: Path,
        run_dir: Optional[Path],
        run_index: int,
        last_iterations: Optional[int],
        filter_trainable_only: bool,
) -> tuple[pd.DataFrame, Path, Path, str]:
    resolved_run_dir = (
        Path(run_dir).resolve()
        if run_dir is not None
        else find_latest_run_dir(
            optimization_output_root=Path(optimization_output_root),
            index=run_index,
        ).resolve()
    )

    per_point_csv_path = resolved_run_dir / "per_point_position_gradient_metrics.csv"

    if not per_point_csv_path.exists():
        raise FileNotFoundError(f"Missing per-point metrics CSV: {per_point_csv_path}")

    per_point_metrics = pd.read_csv(per_point_csv_path)

    # Prefer persistent point_id. Fallback keeps old CSV files usable.
    id_column = "point_id" if "point_id" in per_point_metrics.columns else "point_index"

    required_columns = ["iteration", id_column]
    missing_required_columns = [
        column for column in required_columns
        if column not in per_point_metrics.columns
    ]
    if missing_required_columns:
        raise ValueError(f"Missing required columns: {missing_required_columns}")

    per_point_metrics["iteration"] = pd.to_numeric(
        per_point_metrics["iteration"],
        errors="coerce",
    ).astype("Int64")

    per_point_metrics[id_column] = pd.to_numeric(
        per_point_metrics[id_column],
        errors="coerce",
    ).astype("Int64")

    if "point_index" in per_point_metrics.columns:
        per_point_metrics["point_index"] = pd.to_numeric(
            per_point_metrics["point_index"],
            errors="coerce",
        ).astype("Int64")

    per_point_metrics = per_point_metrics.dropna(
        subset=["iteration", id_column],
    ).copy()

    per_point_metrics["iteration"] = per_point_metrics["iteration"].astype(int)
    per_point_metrics[id_column] = per_point_metrics[id_column].astype(int)

    if "point_index" in per_point_metrics.columns:
        per_point_metrics["point_index"] = per_point_metrics["point_index"].astype(int)

    sort_columns = ["iteration", id_column]
    if "point_index" in per_point_metrics.columns and id_column != "point_index":
        sort_columns.append("point_index")

    per_point_metrics = per_point_metrics.sort_values(
        sort_columns,
    ).reset_index(drop=True)

    if last_iterations is not None and int(last_iterations) > 0:
        max_iteration = int(per_point_metrics["iteration"].max())
        min_iteration_to_keep = max_iteration - int(last_iterations) + 1
        per_point_metrics = per_point_metrics[
            per_point_metrics["iteration"] >= min_iteration_to_keep
        ].copy()

    if filter_trainable_only:
        if "is_trainable" not in per_point_metrics.columns:
            raise ValueError(
                "filter_trainable_only=True, but column 'is_trainable' is missing."
            )

        per_point_metrics["is_trainable"] = pd.to_numeric(
            per_point_metrics["is_trainable"],
            errors="coerce",
        ).fillna(0).astype(int)

        per_point_metrics = per_point_metrics[
            per_point_metrics["is_trainable"] == 1
        ].copy()

        if per_point_metrics.empty:
            raise ValueError("No trainable points found after filtering.")

    return per_point_metrics, resolved_run_dir, per_point_csv_path, id_column


def resolve_columns_to_analyze(
        per_point_metrics: pd.DataFrame,
        selected_columns: Sequence[str],
) -> list[str]:
    columns_to_analyze = [
        column for column in selected_columns
        if column in per_point_metrics.columns
    ]

    if not columns_to_analyze:
        raise ValueError(
            "No selected gradient columns found. "
            f"Available columns: {list(per_point_metrics.columns)}"
        )

    return columns_to_analyze


def select_point_ids_to_plot(
        per_point_metrics: pd.DataFrame,
        id_column: str,
        selected_point_ids: Optional[Sequence[int]],
        top_point_count: int,
        ranking_column: str = "grad_position_total_norm",
) -> list[int]:
    if selected_point_ids is not None:
        return [int(x) for x in selected_point_ids]

    if ranking_column not in per_point_metrics.columns:
        raise ValueError(
            f"selected_point_ids is None, but {ranking_column} is missing. "
            "Set selected_point_ids manually."
        )

    ranking = (
        per_point_metrics
        .assign(
            **{
                ranking_column: pd.to_numeric(
                    per_point_metrics[ranking_column],
                    errors="coerce",
                )
            }
        )
        .groupby(id_column, as_index=False)[ranking_column]
        .max()
        .sort_values(ranking_column, ascending=False)
    )

    return (
        ranking[id_column]
        .head(int(top_point_count))
        .astype(int)
        .tolist()
    )


def build_point_summary(
        filtered_metrics: pd.DataFrame,
        id_column: str,
        columns_to_analyze: Sequence[str],
        albedo_columns: Sequence[str] = ("albedo_r", "albedo_g", "albedo_b"),
) -> pd.DataFrame:
    has_albedo_columns = all(
        column in filtered_metrics.columns
        for column in albedo_columns
    )

    summary_rows = []

    for point_id, point_data in filtered_metrics.groupby(id_column):
        row = {
            id_column: int(point_id),
            "first_iteration": int(point_data["iteration"].min()),
            "last_iteration": int(point_data["iteration"].max()),
            "row_count": int(len(point_data)),
        }

        if "point_birth_iteration" in point_data.columns:
            row["point_birth_iteration"] = int(point_data["point_birth_iteration"].iloc[0])

        if "point_index" in point_data.columns:
            row["first_point_index"] = int(point_data["point_index"].iloc[0])
            row["last_point_index"] = int(point_data["point_index"].iloc[-1])

        if has_albedo_columns:
            latest_albedo = (
                point_data
                .sort_values("iteration")[list(albedo_columns)]
                .iloc[-1]
                .to_numpy(dtype=np.float32)
            )

            if np.nanmax(latest_albedo) > 1.0:
                latest_albedo = latest_albedo / 255.0

            latest_albedo = np.clip(latest_albedo, 0.0, 1.0)

            row["albedo_r"] = float(latest_albedo[0])
            row["albedo_g"] = float(latest_albedo[1])
            row["albedo_b"] = float(latest_albedo[2])
            row["albedo_hex"] = "#{:02x}{:02x}{:02x}".format(
                int(round(255.0 * latest_albedo[0])),
                int(round(255.0 * latest_albedo[1])),
                int(round(255.0 * latest_albedo[2])),
            )

        for column in columns_to_analyze:
            values = pd.to_numeric(point_data[column], errors="coerce")
            row[f"{column}_mean"] = float(values.mean())
            row[f"{column}_max"] = float(values.max())
            row[f"{column}_last"] = float(values.iloc[-1])

        summary_rows.append(row)

    return pd.DataFrame(summary_rows)


def boost_albedo_for_plot(
        rgb,
        assume_linear: bool = True,
        saturation_boost: float = 2.25,
        value_boost: float = 1.15,
        min_value: float = 0.28,
):
    rgb = np.asarray(rgb, dtype=np.float32)

    if not np.all(np.isfinite(rgb)):
        return None

    # Support either [0, 1] or [0, 255].
    if np.nanmax(rgb) > 1.0:
        rgb = rgb / 255.0

    rgb = np.clip(rgb, 0.0, 1.0)

    # Albedos are often linear; Matplotlib colors are display-space.
    if assume_linear:
        rgb = np.power(rgb, 1.0 / 2.2)

    h, s, v = colorsys.rgb_to_hsv(
        float(rgb[0]),
        float(rgb[1]),
        float(rgb[2]),
    )

    s = float(np.clip(s * saturation_boost, 0.0, 1.0))
    v = float(np.clip(max(v * value_boost, min_value), 0.0, 1.0))

    boosted = colorsys.hsv_to_rgb(h, s, v)
    return tuple(float(x) for x in boosted)


def make_point_color_mapper(
        per_point_metrics: pd.DataFrame,
        id_column: str,
        selected_point_ids: Sequence[int],
        albedo_columns: Sequence[str] = ("albedo_r", "albedo_g", "albedo_b"),
        albedo_assume_linear: bool = True,
        albedo_saturation_boost: float = 2.25,
        albedo_value_boost: float = 1.15,
        albedo_min_value: float = 0.28,
):
    has_albedo_columns = all(
        column in per_point_metrics.columns
        for column in albedo_columns
    )

    fallback_color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    fallback_color_by_id = {
        int(point_id): fallback_color_cycle[i % len(fallback_color_cycle)]
        for i, point_id in enumerate(selected_point_ids)
    }

    if has_albedo_columns:
        per_point_metrics = per_point_metrics.copy()

        for column in albedo_columns:
            per_point_metrics[column] = pd.to_numeric(
                per_point_metrics[column],
                errors="coerce",
            )

        albedo_by_id = (
            per_point_metrics
            .sort_values("iteration")
            .groupby(id_column)[list(albedo_columns)]
            .last()
        )
    else:
        albedo_by_id = None

    def color_for_point_id(point_id):
        point_id = int(point_id)

        if albedo_by_id is not None and point_id in albedo_by_id.index:
            rgb = albedo_by_id.loc[point_id, list(albedo_columns)].to_numpy(dtype=np.float32)
            color = boost_albedo_for_plot(
                rgb,
                assume_linear=albedo_assume_linear,
                saturation_boost=albedo_saturation_boost,
                value_boost=albedo_value_boost,
                min_value=albedo_min_value,
            )
            if color is not None:
                return color

        return fallback_color_by_id[point_id]

    return color_for_point_id


def plot_gradient_columns_per_point(
        filtered_metrics: pd.DataFrame,
        id_column: str,
        columns_to_analyze: Sequence[str],
        color_for_point_id,
        figsize=(11, 5),
) -> None:
    for column_name in columns_to_analyze:
        plt.figure(figsize=figsize)

        for point_id, point_data in filtered_metrics.groupby(id_column):
            x_values = point_data["iteration"]
            y_values = pd.to_numeric(point_data[column_name], errors="coerce")

            plt.plot(
                x_values,
                y_values,
                color=color_for_point_id(point_id),
                label=f"{id_column} {point_id}",
            )

        plt.xlabel("iteration")
        plt.ylabel(column_name)
        plt.title(f"{column_name} per {id_column}")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()


def plot_terms_per_point_id(
        filtered_metrics: pd.DataFrame,
        id_column: str,
        selected_point_ids: Sequence[int],
        columns_to_analyze: Sequence[str],
        color_for_point_id,
        figsize=(11, 5),
) -> None:
    line_styles = ["-", "--", "-.", ":"]

    for point_id in selected_point_ids:
        point_data = filtered_metrics[
            filtered_metrics[id_column] == int(point_id)
        ].copy()

        if point_data.empty:
            continue

        base_color = color_for_point_id(point_id)

        plt.figure(figsize=figsize)

        for term_index, column_name in enumerate(columns_to_analyze):
            y_values = pd.to_numeric(point_data[column_name], errors="coerce")

            plt.plot(
                point_data["iteration"],
                y_values,
                color=base_color,
                linestyle=line_styles[term_index % len(line_styles)],
                alpha=max(0.5, 0.95 - 0.12 * term_index),
                label=column_name,
            )

        plt.xlabel("iteration")
        plt.ylabel("gradient norm")
        plt.title(f"Position gradient terms for {id_column} {point_id}")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()


def run_gradient_analyzer(
        optimization_output_root: Path,
        run_dir: Optional[Path] = None,
        run_index: int = 0,
        last_iterations: Optional[int] = 0,
        filter_trainable_only: bool = True,
        selected_point_ids: Optional[Sequence[int]] = None,
        top_point_count: int = 8,
        selected_columns: Sequence[str] = (
            "grad_position_renderer_norm",
            "grad_position_depth_distortion_norm",
            "grad_position_normal_consistency_norm",
            "grad_position_total_norm",
        ),
        ranking_column: str = "grad_position_total_norm",
        albedo_assume_linear: bool = True,
        albedo_saturation_boost: float = 2.25,
        albedo_value_boost: float = 1.15,
        albedo_min_value: float = 0.28,
        plot_terms_per_point: bool = False,
        show_head: bool = True,
        show_summary: bool = True,
):
    per_point_metrics, resolved_run_dir, per_point_csv_path, id_column = load_per_point_metrics(
        optimization_output_root=optimization_output_root,
        run_dir=run_dir,
        run_index=run_index,
        last_iterations=last_iterations,
        filter_trainable_only=filter_trainable_only,
    )

    columns_to_analyze = resolve_columns_to_analyze(
        per_point_metrics=per_point_metrics,
        selected_columns=selected_columns,
    )

    missing_columns = [
        column for column in selected_columns
        if column not in per_point_metrics.columns
    ]

    selected_point_ids = select_point_ids_to_plot(
        per_point_metrics=per_point_metrics,
        id_column=id_column,
        selected_point_ids=selected_point_ids,
        top_point_count=top_point_count,
        ranking_column=ranking_column,
    )

    filtered_metrics = per_point_metrics[
        per_point_metrics[id_column].isin(selected_point_ids)
    ].copy()

    if filtered_metrics.empty:
        raise ValueError(f"No rows found for selected {id_column}s.")

    color_for_point_id = make_point_color_mapper(
        per_point_metrics=per_point_metrics,
        id_column=id_column,
        selected_point_ids=selected_point_ids,
        albedo_assume_linear=albedo_assume_linear,
        albedo_saturation_boost=albedo_saturation_boost,
        albedo_value_boost=albedo_value_boost,
        albedo_min_value=albedo_min_value,
    )

    point_summary = build_point_summary(
        filtered_metrics=filtered_metrics,
        id_column=id_column,
        columns_to_analyze=columns_to_analyze,
    )

    print(f"Run directory : {resolved_run_dir}")
    print(f"Per-point CSV : {per_point_csv_path}")
    print(f"Tracking by   : {id_column}")
    print(f"Rows          : {len(per_point_metrics)}")
    print(f"Tracked IDs   : {per_point_metrics[id_column].nunique()}")
    print(f"Iterations    : {per_point_metrics['iteration'].nunique()}")

    if "point_index" in per_point_metrics.columns and id_column != "point_index":
        print(f"Current point indices in CSV: {per_point_metrics['point_index'].nunique()} unique")

    if missing_columns:
        print("\nMissing selected columns:")
        for column in missing_columns:
            print(f"  {column}")

    print(f"\nSelected {id_column}s:")
    print(selected_point_ids)

    if show_head:
        display(per_point_metrics.head())

    if show_summary:
        display(point_summary)

    plot_gradient_columns_per_point(
        filtered_metrics=filtered_metrics,
        id_column=id_column,
        columns_to_analyze=columns_to_analyze,
        color_for_point_id=color_for_point_id,
    )

    if plot_terms_per_point:
        plot_terms_per_point_id(
            filtered_metrics=filtered_metrics,
            id_column=id_column,
            selected_point_ids=selected_point_ids,
            columns_to_analyze=columns_to_analyze,
            color_for_point_id=color_for_point_id,
        )

    return {
        "per_point_metrics": per_point_metrics,
        "filtered_metrics": filtered_metrics,
        "point_summary": point_summary,
        "selected_point_ids": selected_point_ids,
        "columns_to_analyze": columns_to_analyze,
        "run_dir": resolved_run_dir,
        "csv_path": per_point_csv_path,
        "id_column": id_column,
        "color_for_point_id": color_for_point_id,
    }

In [4]:
# %%
# ============================================================
# Settings + run
# ============================================================

from pathlib import Path

optimization_output_root = Path("../../Assets/OptimizationOutput")

# Set to a concrete run directory if you do not want the latest run.
run_dir = None

# 0 = latest, 1 = second latest, 2 = third latest, ...
run_index = 0

# None or 0 means all iterations.
last_iterations = 0

filter_trainable_only = True

# Pick explicit persistent IDs, e.g. [0, 5, 42].
# If None, the script chooses the top IDs by max grad_position_total_norm.
selected_point_ids = None

top_point_count = 10

selected_columns = [
    "grad_position_renderer_norm",
     "grad_position_depth_distortion_norm",
    # "grad_position_normal_consistency_norm",
    "grad_position_total_norm",
]

plot_terms_per_point = False

# Albedo color display tuning.
albedo_assume_linear = True
albedo_saturation_boost = 2.75
albedo_value_boost = 1.20
albedo_min_value = 0.32

result = run_gradient_analyzer(
    optimization_output_root=optimization_output_root,
    run_dir=run_dir,
    run_index=run_index,
    last_iterations=last_iterations,
    filter_trainable_only=filter_trainable_only,
    selected_point_ids=selected_point_ids,
    top_point_count=top_point_count,
    selected_columns=selected_columns,
    albedo_assume_linear=albedo_assume_linear,
    albedo_saturation_boost=albedo_saturation_boost,
    albedo_value_boost=albedo_value_boost,
    albedo_min_value=albedo_min_value,
    plot_terms_per_point=plot_terms_per_point,
)

EmptyDataError: No columns to parse from file